# 🐕 Muffin vs Chihuahua — Competition Notebook
**6 Models | Auto Submission Format Detection | Best-2 CSV Export**

## 1. Setup & Imports

In [ ]:
import os, warnings, gc, shutil, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import psutil  # for RAM-aware cache decisions
warnings.filterwarnings('ignore')

# ── Step 1: memory growth MUST come first, before ANY GPU op ────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print('GPU memory growth enabled:', gpus[0].name)

# ── Step 2: mixed precision ──────────────────────────────────────────────────
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision:', tf.keras.mixed_precision.global_policy())

# ── Step 3: global XLA ──────────────────────────────────────────────────────
# Do NOT set jit_compile=True per-model on P100 — causes ~1s/step overhead.
tf.config.optimizer.set_jit(True)
print('XLA global JIT:', tf.config.optimizer.get_jit())

print('TensorFlow:', tf.__version__)
print('GPU devices:', gpus)

## 2. Paths & Global Config

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 64
SEED       = 42
VAL_SPLIT  = 0.2
AUTOTUNE   = tf.data.AUTOTUNE

# ── Paths: update competition slug and folder names for your dataset ─────────
COMPETITION_SLUG = 'cs-460-muffin-vs-chihuahua-classification-challenge'  # <-- change this
TRAIN_FOLDER     = 'train'               # folder inside competition input dir
TEST_FOLDER      = 'kaggle_test_final'   # folder inside competition input dir

BASE_INPUT = f'/kaggle/input/competitions/{COMPETITION_SLUG}'
TRAIN_DIR  = os.path.join(BASE_INPUT, TRAIN_FOLDER)
TEST_DIR   = os.path.join(BASE_INPUT, TEST_FOLDER)
WORK_DIR   = '/kaggle/working'
os.makedirs(WORK_DIR, exist_ok=True)

print(f'Train dir : {TRAIN_DIR}  (exists={os.path.isdir(TRAIN_DIR)})')
print(f'Test dir  : {TEST_DIR}   (exists={os.path.isdir(TEST_DIR)})')

In [ ]:
# ── Globals: populated by init_raw_datasets() ────────────────────────────────
_RAW_TRAIN = None
_RAW_VAL   = None
_N_TRAIN   = 0


def _pick_cache_mode(n_images, img_size):
    """Decide RAM / disk / none based on available resources."""
    # Raw uint8 estimate: H*W*3 bytes per image
    bytes_needed = n_images * img_size[0] * img_size[1] * 3
    gb_needed    = bytes_needed / 1e9

    ram_free_gb  = psutil.virtual_memory().available / 1e9
    disk_free_gb = shutil.disk_usage(WORK_DIR).free / 1e9

    print(f'Dataset: {n_images} images  |  ~{gb_needed:.2f} GB raw')
    print(f'RAM free: {ram_free_gb:.1f} GB  |  Disk free: {disk_free_gb:.1f} GB')

    # Keep at least 3 GB RAM headroom for model weights + activations
    if gb_needed < ram_free_gb - 3.0:
        return 'ram'
    # Keep at least 20% disk headroom
    elif gb_needed * 1.2 < disk_free_gb:
        return 'disk'
    else:
        return 'none'


def init_raw_datasets(img_size=IMG_SIZE, force_cache_mode=None):
    """
    Build and (optionally) warm-up the shared raw-image cache.
    Call once before training any model.

    force_cache_mode: override auto-detection with 'ram' | 'disk' | 'none'
    """
    global _RAW_TRAIN, _RAW_VAL, _N_TRAIN

    raw_train = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, validation_split=VAL_SPLIT, subset='training',
        seed=SEED, image_size=img_size, batch_size=None,
    )
    raw_val = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, validation_split=VAL_SPLIT, subset='validation',
        seed=SEED, image_size=img_size, batch_size=None,
    )

    n_train = tf.data.experimental.cardinality(raw_train).numpy()
    n_val   = tf.data.experimental.cardinality(raw_val).numpy()
    _N_TRAIN = n_train

    mode = force_cache_mode or _pick_cache_mode(n_train + n_val, img_size)
    print(f'\nCache mode selected: {mode.upper()}')

    if mode == 'ram':
        _RAW_TRAIN = raw_train.cache()
        _RAW_VAL   = raw_val.cache()
        # Warm up: iterate once so RAM is populated before training starts
        print('Warming up RAM cache (first pass over data)...')
        for _ in _RAW_TRAIN: pass
        for _ in _RAW_VAL:   pass
        print('Cache warm-up done ✅')

    elif mode == 'disk':
        cache_dir = os.path.join(WORK_DIR, '.raw_cache')
        os.makedirs(cache_dir, exist_ok=True)
        _RAW_TRAIN = raw_train.cache(os.path.join(cache_dir, 'train'))
        _RAW_VAL   = raw_val.cache(os.path.join(cache_dir, 'val'))
        print(f'Disk cache at: {cache_dir}')
        # First epoch will be slow (writes cache); subsequent epochs read fast

    else:  # none
        _RAW_TRAIN = raw_train
        _RAW_VAL   = raw_val
        print('No cache — streaming from disk each epoch (GPU may see idle gaps)')

    print(f'Train samples: {n_train}  |  Val samples: {n_val}')


def make_tf_dataset(preprocess_fn, batch_size=BATCH_SIZE):
    """
    Build train/val tf.data pipelines from the shared raw cache.

    Pipeline order:
      raw cache → preprocess (CPU, AUTOTUNE parallel) →
      shuffle → batch → prefetch (CPU/GPU overlap)

    GPU augmentation runs inside the model on the forward pass.
    Call init_raw_datasets() once before using this function.
    """
    assert _RAW_TRAIN is not None, 'Call init_raw_datasets() first.'

    def apply_preprocess(x, y):
        return tf.cast(preprocess_fn(x), tf.float32), y

    shuffle_buf = min(4000, max(int(_N_TRAIN), 100))

    train_ds = (
        _RAW_TRAIN
        .map(apply_preprocess, num_parallel_calls=AUTOTUNE)
        .shuffle(shuffle_buf, seed=SEED, reshuffle_each_iteration=True)
        .batch(batch_size, drop_remainder=True)
        .prefetch(AUTOTUNE)
    )
    val_ds = (
        _RAW_VAL
        .map(apply_preprocess, num_parallel_calls=AUTOTUNE)
        .batch(batch_size)
        .prefetch(AUTOTUNE)
    )
    return train_ds, val_ds


PREPROCESS = {
    'efficientnet':   tf.keras.applications.efficientnet.preprocess_input,
    'mobilenet':      tf.keras.applications.mobilenet_v2.preprocess_input,
    'resnet':         tf.keras.applications.resnet50.preprocess_input,
    'custom_cnn':     lambda x: x / 255.0,
    'efficientnetB4': tf.keras.applications.efficientnet.preprocess_input,
    'efficientnetV2': tf.keras.applications.efficientnet_v2.preprocess_input,
}

print('Dataset builder ready ✅')
print('  → Shared raw cache: one copy for all 6 models (~3× RAM savings)')
print('  → Augmentation runs inside models (GPU forward pass)')
print('  → CPU preprocessing parallelised with AUTOTUNE + prefetch')

## 3. Submission Format Detection
> Reads `sample_submission.csv` (if available) and auto-detects column names, label format, and ID format so the generated CSV always matches what the leaderboard expects.

In [ ]:
def detect_submission_format(base_input_dir):
    """
    Scans common locations for a sample submission file and infers the schema.

    Returns dict with:
        id_col    : name of the ID column
        label_col : name of the label/prediction column
        id_has_ext: bool — does the ID include the image extension?
        label_type: 'string' | 'int' | 'float'
        pos_label : which label value means the POSITIVE class (None if unknown)
        sample_df : the sample_submission dataframe, or None
    """
    schema = dict(
        id_col='ID', label_col='Label',
        id_has_ext=True, label_type='string',
        pos_label=None, sample_df=None
    )

    # Search order: explicit names → glob for any *submission*.csv → working dir
    explicit = [
        os.path.join(base_input_dir, 'sample_submission.csv'),
        os.path.join(base_input_dir, 'test_solution_01.csv'),
        '/kaggle/input/sample_submission.csv',
        '/kaggle/working/sample_submission.csv',
    ]
    # Also glob any *submission*.csv anywhere under the input directory
    glob_hits = glob.glob(os.path.join(base_input_dir, '**', '*submission*.csv'),
                          recursive=True)
    glob_hits += glob.glob(os.path.join(base_input_dir, '**', '*solution*.csv'),
                           recursive=True)

    candidates = explicit + [p for p in glob_hits if p not in explicit]
    sample_path = next((p for p in candidates if os.path.isfile(p)), None)

    if sample_path:
        df = pd.read_csv(sample_path)
        schema['sample_df'] = df
        cols = list(df.columns)
        print(f'✅  sample_submission found: {sample_path}')
        print(f'   Columns  : {cols}')
        print(f'   First row: {df.iloc[0].to_dict()}')

        schema['id_col']    = cols[0]
        schema['label_col'] = cols[-1]

        first_id = str(df.iloc[0, 0])
        schema['id_has_ext'] = any(
            first_id.lower().endswith(ext)
            for ext in ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
        )

        lbl_dtype = df[schema['label_col']].dtype
        if np.issubdtype(lbl_dtype, np.integer):
            schema['label_type'] = 'int'
        elif np.issubdtype(lbl_dtype, np.floating):
            schema['label_type'] = 'float'
        else:
            schema['label_type'] = 'string'
    else:
        print('⚠️  No sample_submission.csv found — using defaults (ID, Label).')

    print(f'\n📋 Schema: {schema}')
    return schema


SCHEMA = detect_submission_format(BASE_INPUT)

## 4. Class-Order Verification
> **Root cause of the 52% bug** — Keras sorts class names alphabetically, so `chihuahua=0, muffin=1`. We pin this mapping explicitly so it never drifts.

In [ ]:
# Build a temp dataset just to read the class order — never train on it.
_probe_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=1, seed=SEED
)
CLASS_NAMES = _probe_ds.class_names   # e.g. ['chihuahua', 'muffin']
del _probe_ds

print('Class names (alphabetical):', CLASS_NAMES)
# Sigmoid output → index 1 class → muffin (if alphabetical order holds)
# Index mapping: 0 → CLASS_NAMES[0], 1 → CLASS_NAMES[1]

def sigmoid_to_label(prob, threshold=0.5):
    """Map sigmoid probability to string label."""
    return CLASS_NAMES[1] if prob >= threshold else CLASS_NAMES[0]

print(f'\nsigmoid ≥ 0.5  →  "{CLASS_NAMES[1]}"')
print(f'sigmoid <  0.5  →  "{CLASS_NAMES[0]}"')

In [ ]:
init_raw_datasets(IMG_SIZE)

## 6. Shared Training Utilities

In [ ]:
# Shared GPU augmentation block — runs on GPU during the forward pass.
GPU_AUGMENT = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1),
    layers.RandomTranslation(0.05, 0.05),
], name='gpu_augmentation')


def build_head(base_model, name):
    inp = tf.keras.Input(shape=(*IMG_SIZE, 3), name='image_input')
    x   = GPU_AUGMENT(inp)                  # augment on GPU
    x   = base_model(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid',
                       dtype='float32', name='output')(x)  # float32 for numerical stability
    return tf.keras.Model(inp, out, name=name)


def compile_model(model, lr=1e-3):
    """No jit_compile — P100 CC 6.0 XLA retracing cost > benefit."""
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=['accuracy'],
    )


def get_callbacks(model_name, patience_es=5, patience_lr=3):
    ckpt_path = f'{WORK_DIR}/{model_name}_best.keras'
    return [
        callbacks.EarlyStopping(
            monitor='val_accuracy', patience=patience_es,
            restore_best_weights=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.3, patience=patience_lr,
            min_lr=1e-7, verbose=1
        ),
        callbacks.ModelCheckpoint(
            ckpt_path, monitor='val_accuracy',
            save_best_only=True, verbose=0
        ),
    ]


def log_gpu_memory(label=''):
    """Print GPU VRAM usage — helpful to catch memory creep between models."""
    import subprocess
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        for line in r.stdout.strip().splitlines():
            name, util, used, total = [v.strip() for v in line.split(',')]
            print(f'  [{label}] {name}  util={util}%  VRAM={used}/{total} MiB'
                  + ('  ⚠️  >80%' if int(used)/int(total) > 0.8 else ''))
    ram = psutil.virtual_memory()
    print(f'  [{label}] RAM used={ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB'
          + ('  ⚠️  >80%' if ram.percent > 80 else ''))


def plot_history(histories, names):
    n = len(histories)
    fig, axes = plt.subplots(2, n, figsize=(5*n, 8))
    for i, (hist, name) in enumerate(zip(histories, names)):
        axes[0, i].plot(hist.history['accuracy'],     label='train')
        axes[0, i].plot(hist.history['val_accuracy'], label='val')
        axes[0, i].set_title(f'{name}\nAccuracy')
        axes[0, i].legend(); axes[0, i].grid(True)
        axes[1, i].plot(hist.history['loss'],     label='train')
        axes[1, i].plot(hist.history['val_loss'], label='val')
        axes[1, i].set_title(f'{name}\nLoss')
        axes[1, i].legend(); axes[1, i].grid(True)
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/training_curves.png', dpi=120)
    plt.show()


print('Model builder & utilities ready ✅')
print('Expected step time: ~130-200 ms (vs 1000 ms with jit_compile)')

In [ ]:
# Sanity check: augmentation runs on GPU
with tf.device('/GPU:0'):
    dummy = tf.random.normal([4, 224, 224, 3])
    out   = GPU_AUGMENT(dummy, training=True)
print('Augmentation device check OK, output shape:', out.shape)
log_gpu_memory('post-setup')

## 7. Model 1 — EfficientNetB0

In [ ]:
train_ds1, val_ds1 = make_tf_dataset(PREPROCESS['efficientnet'])

base1 = tf.keras.applications.EfficientNetB0(
    include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3)
)
base1.trainable = False
model1 = build_head(base1, 'Model1_EfficientNetB0')
compile_model(model1, lr=1e-3)

print('=== Model 1 — Phase 1: Frozen backbone ===')
history1_p1 = model1.fit(
    train_ds1, validation_data=val_ds1, epochs=10,
    callbacks=get_callbacks('model1_p1', patience_es=4)
)

print('\n=== Model 1 — Phase 2: Fine-tuning top 30 layers ===')
base1.trainable = True
for layer in base1.layers[:-30]: layer.trainable = False
compile_model(model1, lr=1e-5)
history1_p2 = model1.fit(
    train_ds1, validation_data=val_ds1, epochs=10,
    callbacks=get_callbacks('model1_p2', patience_es=5)
)

val_acc1 = max(history1_p2.history['val_accuracy'])
del train_ds1, val_ds1  # release pipeline references; shared cache stays
gc.collect()
log_gpu_memory('after model1')
print(f'\n✅ Model 1 best val_accuracy: {val_acc1:.4f}')

## 8. Model 2 — MobileNetV2

In [ ]:
train_ds2, val_ds2 = make_tf_dataset(PREPROCESS['mobilenet'])

base2 = tf.keras.applications.MobileNetV2(
    include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3)
)
base2.trainable = False
model2 = build_head(base2, 'Model2_MobileNetV2')
compile_model(model2, lr=1e-3)

print('=== Model 2 — Phase 1: Frozen backbone ===')
history2_p1 = model2.fit(
    train_ds2, validation_data=val_ds2, epochs=10,
    callbacks=get_callbacks('model2_p1', patience_es=4)
)

print('\n=== Model 2 — Phase 2: Fine-tuning top 30 layers ===')
base2.trainable = True
for layer in base2.layers[:-30]: layer.trainable = False
compile_model(model2, lr=1e-5)
history2_p2 = model2.fit(
    train_ds2, validation_data=val_ds2, epochs=10,
    callbacks=get_callbacks('model2_p2', patience_es=5)
)

val_acc2 = max(history2_p2.history['val_accuracy'])
del train_ds2, val_ds2
gc.collect()
log_gpu_memory('after model2')
print(f'\n✅ Model 2 best val_accuracy: {val_acc2:.4f}')

## 9. Model 3 — ResNet50

In [ ]:
train_ds3, val_ds3 = make_tf_dataset(PREPROCESS['resnet'])

base3 = tf.keras.applications.ResNet50(
    include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3)
)
base3.trainable = False
model3 = build_head(base3, 'Model3_ResNet50')
compile_model(model3, lr=1e-3)

print('=== Model 3 — Phase 1: Frozen backbone ===')
history3_p1 = model3.fit(
    train_ds3, validation_data=val_ds3, epochs=10,
    callbacks=get_callbacks('model3_p1', patience_es=4)
)

print('\n=== Model 3 — Phase 2: Fine-tuning last ResNet block ===')
base3.trainable = True
for layer in base3.layers[:-35]: layer.trainable = False
compile_model(model3, lr=1e-5)
history3_p2 = model3.fit(
    train_ds3, validation_data=val_ds3, epochs=10,
    callbacks=get_callbacks('model3_p2', patience_es=5)
)

val_acc3 = max(history3_p2.history['val_accuracy'])
del train_ds3, val_ds3
gc.collect()
log_gpu_memory('after model3')
print(f'\n✅ Model 3 best val_accuracy: {val_acc3:.4f}')

## 10. Model 4 — Custom CNN

In [ ]:
train_ds4, val_ds4 = make_tf_dataset(PREPROCESS['custom_cnn'])

inp4 = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = layers.Conv2D(32, 3, padding='same', activation='relu')(inp4)
x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.25)(x)

x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.25)(x)

x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.3)(x)

x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
out4 = layers.Dense(1, activation='sigmoid', dtype='float32')(x)
model4 = tf.keras.Model(inp4, out4, name='Model4_CustomCNN')

compile_model(model4, lr=1e-3)
print('=== Model 4 — Custom CNN ===')
history4 = model4.fit(
    train_ds4, validation_data=val_ds4, epochs=20,
    callbacks=get_callbacks('model4', patience_es=6, patience_lr=4)
)

val_acc4 = max(history4.history['val_accuracy'])
del train_ds4, val_ds4
gc.collect()
log_gpu_memory('after model4')
print(f'\n✅ Model 4 best val_accuracy: {val_acc4:.4f}')

## 11. Model 5 — EfficientNetB4

In [ ]:
train_ds5, val_ds5 = make_tf_dataset(PREPROCESS['efficientnetB4'], batch_size=32)

base5 = tf.keras.applications.EfficientNetB4(
    include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3)
)
base5.trainable = False
model5 = build_head(base5, 'Model5_EfficientNetB4')
compile_model(model5, lr=1e-3)

print('=== Model 5 — Phase 1: Frozen backbone ===')
history5_p1 = model5.fit(
    train_ds5, validation_data=val_ds5, epochs=10,
    callbacks=get_callbacks('model5_p1', patience_es=4)
)

print('\n=== Model 5 — Phase 2: Fine-tuning top 40 layers ===')
base5.trainable = True
for layer in base5.layers[:-40]: layer.trainable = False
compile_model(model5, lr=1e-5)
history5_p2 = model5.fit(
    train_ds5, validation_data=val_ds5, epochs=10,
    callbacks=get_callbacks('model5_p2', patience_es=5)
)

val_acc5 = max(history5_p2.history['val_accuracy'])
del train_ds5, val_ds5
gc.collect()
log_gpu_memory('after model5')
print(f'\n✅ Model 5 best val_accuracy: {val_acc5:.4f}')

## 12. Model 6 — EfficientNetV2-S

In [ ]:
train_ds6, val_ds6 = make_tf_dataset(PREPROCESS['efficientnetV2'], batch_size=32)

base6 = tf.keras.applications.EfficientNetV2S(
    include_top=False, weights='imagenet', input_shape=(*IMG_SIZE, 3)
)
base6.trainable = False
model6 = build_head(base6, 'Model6_EfficientNetV2S')
compile_model(model6, lr=1e-3)

print('=== Model 6 — Phase 1: Frozen backbone ===')
history6_p1 = model6.fit(
    train_ds6, validation_data=val_ds6, epochs=10,
    callbacks=get_callbacks('model6_p1', patience_es=4)
)

print('\n=== Model 6 — Phase 2: Fine-tuning top 40 layers ===')
base6.trainable = True
for layer in base6.layers[:-40]: layer.trainable = False
compile_model(model6, lr=1e-5)
history6_p2 = model6.fit(
    train_ds6, validation_data=val_ds6, epochs=10,
    callbacks=get_callbacks('model6_p2', patience_es=5)
)

val_acc6 = max(history6_p2.history['val_accuracy'])
del train_ds6, val_ds6
gc.collect()
log_gpu_memory('after model6')
print(f'\n✅ Model 6 best val_accuracy: {val_acc6:.4f}')

## 11. Model Selection — Best 2

In [ ]:
model_registry = [
    {'name': 'Model1_EfficientNetB0',  'model': model1, 'val_acc': val_acc1, 'preprocess': PREPROCESS['efficientnet']},
    {'name': 'Model2_MobileNetV2',     'model': model2, 'val_acc': val_acc2, 'preprocess': PREPROCESS['mobilenet']},
    {'name': 'Model3_ResNet50',        'model': model3, 'val_acc': val_acc3, 'preprocess': PREPROCESS['resnet']},
    {'name': 'Model4_CustomCNN',       'model': model4, 'val_acc': val_acc4, 'preprocess': PREPROCESS['custom_cnn']},
    {'name': 'Model5_EfficientNetB4',  'model': model5, 'val_acc': val_acc5, 'preprocess': PREPROCESS['efficientnetB4']},
    {'name': 'Model6_EfficientNetV2S', 'model': model6, 'val_acc': val_acc6, 'preprocess': PREPROCESS['efficientnetV2']},
]

# Sort by validation accuracy and pick top 2
model_registry.sort(key=lambda m: m['val_acc'], reverse=True)

print('🏆 Model Rankings:')
for rank, m in enumerate(model_registry, 1):
    tag = '🥇' if rank == 1 else '🥈' if rank == 2 else '  '
    print(f'  {tag} Rank {rank}: {m["name"]}  —  val_accuracy = {m["val_acc"]:.4f}')

best_two = model_registry[:2]
print(f'\n📌 Generating submissions for: {[m["name"] for m in best_two]}')

## 12. Build Test Tensor

In [ ]:
VALID_EXT = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

test_paths = sorted([
    os.path.join(TEST_DIR, f)
    for f in os.listdir(TEST_DIR)
    if f.lower().endswith(VALID_EXT)
])

raw_fnames = [os.path.basename(p) for p in test_paths]
if SCHEMA['id_has_ext']:
    test_ids = raw_fnames
else:
    test_ids = [os.path.splitext(f)[0] for f in raw_fnames]

print(f'Total test images : {len(test_paths)}')
print(f'ID format (sample): {test_ids[:3]}')
log_gpu_memory('pre-inference')


def make_test_dataset(paths, preprocess_fn,
                      img_size=IMG_SIZE, batch_size=32):
    """
    Memory-safe test pipeline: loads + preprocesses images on-the-fly.
    Peak memory = one batch, not the full test set.
    Works for any dataset size.
    """
    def load_and_preprocess(path):
        raw = tf.io.read_file(path)
        # decode_image handles jpg/png/bmp/webp automatically
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32)
        return preprocess_fn(img)

    return (
        tf.data.Dataset.from_tensor_slices(paths)
        .map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        .batch(batch_size)
        .prefetch(AUTOTUNE)
    )

print('Test pipeline ready ✅  (streaming — no full-dataset numpy array)')

## 13. Prediction with Test-Time Augmentation (TTA)

In [ ]:
def predict_with_tta(model, paths, preprocess_fn,
                     n_tta=5, batch_size=32, img_size=IMG_SIZE):
    """
    Test-Time Augmentation using a streaming tf.data pipeline.

    - Each TTA pass re-loads images with random augmentation applied.
    - Peak memory ≈ one batch × n_tta float arrays (not the full test set).
    - Returns probability array of shape (N,).
    """
    tta_aug = tf.keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.05),
    ], name='tta_aug')

    def load_aug_preprocess(path):
        raw = tf.io.read_file(path)
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32)
        img = tta_aug(img, training=True)   # random augmentation per pass
        return preprocess_fn(img)

    all_probs = []
    for t in range(n_tta):
        ds = (
            tf.data.Dataset.from_tensor_slices(paths)
            .map(load_aug_preprocess, num_parallel_calls=AUTOTUNE)
            .batch(batch_size)
            .prefetch(AUTOTUNE)
        )
        probs = model.predict(ds, verbose=0)
        all_probs.append(probs.flatten())
        print(f'  TTA pass {t+1}/{n_tta} done', end='\r')

    print()  # newline after \r
    return np.mean(all_probs, axis=0)   # shape (N,)

print('TTA prediction utility ready ✅')

## 14. Generate Submissions for Best 2 Models

In [ ]:
def build_submission(probs, test_ids, schema, threshold=0.5):
    """
    Convert probability array → properly-formatted submission DataFrame.
    Handles string labels, integer (0/1), and float probability formats.
    """
    label_type = schema['label_type']

    if label_type == 'string':
        labels = [sigmoid_to_label(p, threshold) for p in probs]
    elif label_type == 'int':
        labels = [(1 if p >= threshold else 0) for p in probs]
    elif label_type == 'float':
        labels = probs.tolist()
    else:
        labels = [sigmoid_to_label(p, threshold) for p in probs]

    df = pd.DataFrame({
        schema['id_col']:    test_ids,
        schema['label_col']: labels,
    })

    # Sanity checks
    assert len(df) == len(test_ids),                    'Row count mismatch!'
    assert df[schema['id_col']].nunique() == len(df),   'Duplicate IDs found!'
    assert df[schema['label_col']].notna().all(),        'NaN labels found!'

    return df


output_paths = []

for rank, entry in enumerate(best_two, 1):
    mname = entry['name']
    mdl   = entry['model']
    prep  = entry['preprocess']
    vacc  = entry['val_acc']

    print(f'\n━━━ Rank {rank}: {mname}  (val_acc={vacc:.4f}) ━━━')
    probs = predict_with_tta(mdl, test_paths, prep, n_tta=5, batch_size=32)

    df_sub = build_submission(probs, test_ids, SCHEMA)

    out_path = f'{WORK_DIR}/submission_rank{rank}_{mname}.csv'
    df_sub.to_csv(out_path, index=False)
    output_paths.append(out_path)

    print(f'✅ Saved: {out_path}')
    print(f'   Rows  : {len(df_sub)}')
    print(f'   Cols  : {list(df_sub.columns)}')
    print(f'   Dist  : {df_sub[SCHEMA["label_col"]].value_counts().to_dict()}')
    print(df_sub.head(5).to_string(index=False))

    gc.collect()  # clean up between predictions

print('\n' + '='*60)
print('📦 Generated submission files:')
for p in output_paths:
    print(f'   → {p}')
       

## 15. Submission Verification

In [ ]:
def verify_submission(csv_path, schema, expected_rows=None):
    """
    Full sanity-check on a generated submission CSV.
    Prints PASS / FAIL for each check.
    """
    print(f'\n🔍 Verifying: {os.path.basename(csv_path)}')
    df = pd.read_csv(csv_path)
    checks = {}

    checks['correct_columns'] = (
        schema['id_col']    in df.columns and
        schema['label_col'] in df.columns
    )
    checks['no_nulls']         = df.notna().all().all()
    checks['no_duplicate_ids'] = df[schema['id_col']].nunique() == len(df)

    if expected_rows:
        checks['row_count_correct'] = len(df) == expected_rows

    vc = df[schema['label_col']].value_counts(normalize=True)
    checks['not_all_one_class'] = (vc.max() < 0.95)

    sample_id = str(df[schema['id_col']].iloc[0])
    has_ext   = any(sample_id.lower().endswith(e)
                    for e in ('.jpg', '.jpeg', '.png', '.webp', '.bmp'))
    checks['id_format_correct'] = (has_ext == schema['id_has_ext'])

    all_pass = True
    for chk, result in checks.items():
        icon = '✅' if result else '❌'
        print(f'  {icon} {chk}')
        if not result:
            all_pass = False

    print(f'  Label distribution: {df[schema["label_col"]].value_counts().to_dict()}')
    print(f'  Total rows: {len(df)}')
    print(f'  → Overall: {"✅ PASS" if all_pass else "❌ FAIL"}')
    return all_pass


print('\n' + '='*60)
print('SUBMISSION VERIFICATION REPORT')
print('='*60)
for path in output_paths:
    verify_submission(path, SCHEMA, expected_rows=len(test_ids))

## 16. Training Curves

In [ ]:
# Merge phase1+phase2 histories for pretrained models
def merge_histories(h1, h2):
    merged = {k: h1.history[k] + h2.history[k] for k in h1.history}
    class FakeHistory:
        def __init__(self, d): self.history = d
    return FakeHistory(merged)

all_histories = [
    merge_histories(history1_p1, history1_p2),
    merge_histories(history2_p1, history2_p2),
    merge_histories(history3_p1, history3_p2),
    history4,
    merge_histories(history5_p1, history5_p2),
    merge_histories(history6_p1, history6_p2),
]
all_names = [
    f'Model1\nEfficientNetB0\n({val_acc1:.3f})',
    f'Model2\nMobileNetV2\n({val_acc2:.3f})',
    f'Model3\nResNet50\n({val_acc3:.3f})',
    f'Model4\nCustomCNN\n({val_acc4:.3f})',
    f'Model5\nEfficientNetB4\n({val_acc5:.3f})',
    f'Model6\nEfficientNetV2S\n({val_acc6:.3f})',
]
plot_history(all_histories, all_names)

## 17. Summary

In [ ]:
print('='*60)
print('FINAL SUMMARY')
print('='*60)
for rank, m in enumerate(model_registry, 1):
    tag = '🥇 BEST ' if rank==1 else '🥈 2nd  ' if rank==2 else '       '
    print(f'{tag} {m["name"]:35s}  val_acc={m["val_acc"]:.4f}')
print()
print('Submission CSVs:')
for path in output_paths:
    print(f'  → {path}')
print()
print('Submission column format:')
print(f'  ID col    : {SCHEMA["id_col"]}')
print(f'  Label col : {SCHEMA["label_col"]}')
print(f'  Label type: {SCHEMA["label_type"]}')
print(f'  ID has ext: {SCHEMA["id_has_ext"]}')
print()
print('Class mapping (sigmoid output):')
print(f'  p >= 0.5  →  "{CLASS_NAMES[1]}"')
print(f'  p <  0.5  →  "{CLASS_NAMES[0]}"')
print()
log_gpu_memory('final')